In [2]:
from langgraph.graph import StateGraph, START, END            
from typing import TypedDict
from langchain_ollama import ChatOllama  

In [3]:
model = ChatOllama(model="llama3.2:latest")

In [4]:
class LLMState(TypedDict):
    topic: str
    outline: str
    blog: str

In [5]:
def write_outline(state: LLMState) -> LLMState:

    topic = state['topic']

    outline = model.invoke(f'give a short outline on the following topic : {topic}').content

    state['outline'] = outline

    return state


In [6]:
def write_blog(state: LLMState) -> LLMState:

    topic = state['topic']
    outline = state['outline']

    blog = model.invoke(f'Write a short blog on the topic : {topic} its outline is as follows : {outline}').content

    state['blog'] = blog

    return state

In [7]:
graph = StateGraph(LLMState)

#  adding nodes
graph.add_node('write_outline', write_outline)
graph.add_node('write_blog', write_blog)

# adding edges

graph.add_edge(START, 'write_outline')
graph.add_edge('write_outline', 'write_blog')
graph.add_edge('write_blog', END)

workflow = graph.compile()

In [8]:
initial_state = {'topic': "indian youth"}

final_state = workflow.invoke(initial_state)

In [9]:
print(final_state)

{'topic': 'indian youth', 'outline': 'Here is a short outline on the topic "Indian Youth":\n\nI. Introduction\n\n* Definition: Indian Youth refers to individuals between the ages of 15 and 30 years old in India.\n* Significance: Indian Youth plays a crucial role in shaping the country\'s future and economy.\n\nII. Demographics\n\n* Population: Over 300 million (as of 2022)\n* Age distribution: 60% between 15-24 years old, 30% between 25-34 years old\n* Urbanization: Rapid urbanization, with 40% of the youth living in urban areas\n\nIII. Education and Skills\n\n* Education: 60% of Indian youth are illiterate or have low literacy rates\n* Skills: Growing demand for technical and vocational skills, with a focus on areas like IT, healthcare, and renewable energy\n* Challenges: Limited access to quality education and training, particularly in rural areas\n\nIV. Employment and Economic Opportunities\n\n* Unemployment: High rates of unemployment, particularly among youth, with around 30% of t